# OncoReg Studio — OpenAI 연동 실행 노트북

자유서술 검색 · 근거수준 가중 랭킹 · 문서 스튜디오(오른쪽 실제 서식 + 왼쪽 근거 삽입).

**흐름**: ⓪초기화 → ①설치 → ②파일 기록 → ③OpenAI 키 입력 → ④서버 실행(cloudflared URL).

> ⚠️ 문헌·수치·지정(긴급승인·희귀의약품)·가이드라인은 AI 생성 초안이라 실제와 다를 수 있습니다. 제출 전 반드시 원문·규제정보 검증이 필요합니다.

> 🔴 다른 앱과 안 섞이게 전용 폴더(`oncoreg_studio/`)·모듈명(`studio_app`)·포트(8020)를 씁니다. 섞이면 런타임 다시 시작 후 이 노트북만 실행하세요.

## 0) 초기화

In [ ]:
import sys
for _m in ['server','studio_app','oncoreg_app','trialmatch_app']:
    sys.modules.pop(_m, None)
print('모듈 캐시 정리 완료.')


## 1) 패키지 설치

In [ ]:
!pip -q install google-genai openai flask flask-cors flask-cloudflared


## 2) 백엔드/프론트 파일 기록 (전용 폴더 `oncoreg_studio/`)

In [ ]:
import os; os.makedirs('oncoreg_studio/templates', exist_ok=True); print('준비 완료')


In [ ]:
%%writefile oncoreg_studio/studio_app.py
"""
OncoReg Studio — 자유서술/PICO 검색 + 상세 유사도 가중 랭킹 + 문서 스튜디오
공급자 자동감지: GEMINI_API_KEY 있으면 Gemini, 없고 OPENAI_API_KEY 있으면 OpenAI.

엔드포인트
  - /            : templates/index.html
  - /api/health  : 상태(provider/model/key)
  - /api/search  : {text} 또는 {pico:{p,i,c,o}} → {query(PICO 포함), papers, weights}
  - /api/drug    : {name, kind} → 지정(긴급승인·희귀의약품)·대상·가이드라인·근거 논문

주의: 문헌·수치·지정·가이드라인은 LLM 생성 초안으로 실제와 다를 수 있다(반드시 원문/공고 검증).
로컬:  GEMINI_API_KEY=... python server.py   ->  http://localhost:8000
"""
import os
import json
from flask import Flask, request, jsonify, render_template
from flask_cors import CORS

# 유사도 가중치 축(사용자 지정) + 근거수준. 합=1.
DEFAULT_WEIGHTS = {"genetic": 0.22, "symptom": 0.15, "age": 0.10,
                   "comorbidity": 0.14, "pathology": 0.17, "priormed": 0.14, "evidence": 0.08}
SIM_AXES = ["genetic", "symptom", "age", "comorbidity", "pathology", "priormed"]

EVIDENCE_LEVELS = ["체계적 문헌고찰/메타분석", "무작위 대조연구(RCT)", "코호트 연구",
                   "환자-대조군 연구", "사례군/사례보고"]


# ---------------------------------------------------------------- LLM 공급자
def _provider() -> str:
    p = os.environ.get("LLM_PROVIDER", "").lower()
    if p in ("gemini", "openai"):
        return p
    if os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY"):
        return "gemini"
    if os.environ.get("OPENAI_API_KEY"):
        return "openai"
    return "gemini"


def _model() -> str:
    return (os.environ.get("GEMINI_MODEL", "gemini-2.5-flash")
            if _provider() == "gemini" else os.environ.get("OPENAI_MODEL", "gpt-4o-mini"))


def _key_present() -> bool:
    return (bool(os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY"))
            if _provider() == "gemini" else bool(os.environ.get("OPENAI_API_KEY")))


def chat_json(system: str, user: str, temperature=0.6) -> dict:
    if _provider() == "gemini":
        from google import genai
        from google.genai import types
        key = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
        if not key:
            raise RuntimeError("GEMINI_API_KEY 환경변수가 설정되지 않았습니다.")
        client = genai.Client(api_key=key)
        kwargs = dict(response_mime_type="application/json", temperature=temperature)
        # 속도 조절: 2.5 flash 계열은 기본 'thinking'이 켜져 느리다.
        #   GEMINI_THINKING=0(기본, 빠름) / 양수=사고 예산 늘려 품질↑(느려짐).
        try:
            if "flash" in _model().lower():
                tb = int(os.environ.get("GEMINI_THINKING", "0"))
                kwargs["thinking_config"] = types.ThinkingConfig(thinking_budget=tb)
        except Exception:
            pass
        resp = client.models.generate_content(
            model=_model(), contents=system + "\n\n" + user,
            config=types.GenerateContentConfig(**kwargs))
        return json.loads(resp.text)
    from openai import OpenAI
    key = os.environ.get("OPENAI_API_KEY")
    if not key:
        raise RuntimeError("OPENAI_API_KEY 환경변수가 설정되지 않았습니다.")
    client = OpenAI(api_key=key)
    r = client.chat.completions.create(
        model=_model(),
        messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
        response_format={"type": "json_object"}, temperature=temperature)
    return json.loads(r.choices[0].message.content)


# ---------------------------------------------------------------- 프롬프트
SEARCH_SYS = """당신은 근거중심의학 리서치 보조자다. 사용자의 환자 상황(자유서술 또는 PICO)을
해석해, 허가초과 사용승인 서류에 쓸 근거 문헌을 '풍부하게' 구조화한다. 반드시 아래 JSON
'하나의 객체'로만 답한다. 한국어.

{
 "query": {
   "P":"대상 환자·질환(짧게)", "I":"중재(약물/시술)", "C":"비교 대상", "O":"평가 결과지표",
   "condition":"핵심 질환명", "drug":"검토 약제", "biomarker":"유전체/바이오마커",
   "stage":"병기/중증도", "line":"이전 치료력", "excess_type":"eff|dose|age", "rare":true/false,
   "special": true/false,          // 특이/복잡 케이스면 true, 전형적이면 false
   "summary_ko":"1~2문장 요약"
 },
 "papers": [                        // 6~8편. 근거수준을 섞고, 유사도가 서로 다르게.
   {
     "id":1, "title":"제목(한국어)", "journal":"출처", "year":2023, "n":정수 또는 null,
     "doi":"있으면 DOI, 없으면 \\"\\"",
     "evidence_level":"체계적 문헌고찰/메타분석|무작위 대조연구(RCT)|코호트 연구|환자-대조군 연구|사례군/사례보고",
     "evidence_rank":1,             // 1(최상)~5(최하)
     "sim":{"genetic":0-100,"symptom":0-100,"age":0-100,"comorbidity":0-100,"pathology":0-100,"priormed":0-100},
     "sim_detail":{                 // 각 축이 '왜 그 점수인지' 한 구절씩(구체적으로)
       "genetic":"예: 동일 HER2 IHC 1+ 집단, KRAS 상태 유사",
       "symptom":"","age":"예: 중앙연령 58세로 유사","comorbidity":"","pathology":"","priormed":"예: 이전 항암 2차 경험 일치"
     },
     "relevance":0-100,             // 이 케이스의 '문서작성 유용성'을 스스로 종합 판단한 점수.
                                    //   진료지침/직접 해당하는 RCT=높게(85~100), 부수적=낮게. 정렬 기준.
     "why":"이 논문이 본 환자와 얼마나/왜 유사·적합한지 2~3문장(무엇이 같고 무엇이 다른지 구체적으로)",
     "items":[                       // 서식 각 칸에 넣을 '근거 조각'. 반드시 원문 한 문장·위치·대상 칸 포함.
       {"key":"ORR","label":"객관적 반응률","value":"52.6%",
        "pre":"영어 원문 앞부분 ","mark":"the ORR was 52.6%","post":" ...(원문 문장 끝).",
        "loc":"Results · Table 2",
        "field":"evidence"          // 이 항목이 들어갈 서식 칸(아래 목록 중 하나)
       }
     ]
   }
 ]
}

[items.field 배정 규칙] 각 근거 항목을 '알맞은 서식 칸' 하나로 분류한다:
- "evidence"  : 유효성·안전성 수치(반응률·생존·위험비·이상반응 발생률 등) → 의학적 근거자료
- "merits"    : 약제의 특장점·기전·표준 대비 우월성·권고 위상 → 신청약제의 특·장점
- "target"    : 대상 환자·선정 기준·바이오마커 정의 → 대상 환자 기준
- "dosage"    : 용법·용량·투여 스케줄 → 용법·용량
- "duration"  : 투여기간·투여중단 시점 → 투여기간
- "other"     : 재투여·모니터링 기준 → 기타(투여방법)
- "reason2"   : 대체약제 부재/우월성 등 고시 제2조 사유 → 고시 제2조 해당 사유
- "opinion"   : 그 외 참고 의견 → 기타 의견
  잘 모르면 "evidence".

규칙:
- **relevance 는 사용자 케이스에 대한 '문서작성 유용성'을 스스로 판단해 매긴다**. 진료지침·직접
  해당하는 RCT는 85~100, 부분적으로만 겹치면 낮게. (사용자에게 가중치를 묻지 않는다 — 모델이 판단)
- papers 6~8편, 근거수준 다양(메타분석~사례보고). 같은 질환이라도 매번 조금씩 다른 논문을
  제시해도 좋다(temperature 반영). 희귀질환이면 사례군/사례보고도 포함.
- 각 논문 items 2~5개로 '여러 칸'을 채우도록 field 를 다양하게(유효성·특장점·대상·용법 등).
  안전성 item 1개 이상(key 를 "AE"로, field 는 "evidence").
- pre/mark/post 는 실제 논문에 나올 법한 '영어 원문 한 문장'으로, mark 안에 value 가 그대로 포함.
- sim 6축을 냉정하게 매기고, sim_detail 로 근거를 남긴다. 과장 금지, 검증 전 초안."""

DRUG_SYS = """당신은 의약품 규제·급여 정보를 정리하는 의학 보조자다. 준 약제(또는 질환)에 대해
허가초과/희귀의약품/긴급(신속)승인 관점 정보를 구조화한다. 아래 JSON '하나의 객체'로만. 한국어.
확실치 않으면 '확인 필요'라고 쓰고 confidence 를 낮춘다.

{
 "drug":"약제명", "disease":"관련 질환",
 "designation":{"orphan":"희귀의약품 지정 여부/대상","emergency":"긴급·신속승인/특례","note":"주의"},
 "populations":["대표 환자군 2~4개"],
 "guidelines":[{"name":"","org":"","year":2024,"recommendation":""}],
 "papers":[ (search 와 동일한 paper 객체, 3~5편) ],
 "confidence":"high|medium|low"
}
규칙: 규제 사실은 단정 말고 '확인 필요' 적극 사용. papers 는 sim 6축·sim_detail·why·items 포함."""


# ---------------------------------------------------------------- 정합성 보정
def sanitize_papers(papers):
    out = []
    for i, p in enumerate(papers or []):
        p["id"] = p.get("id", i + 1)
        sim = p.get("sim") or {}
        p["sim"] = {a: _clamp(sim.get(a, 55)) for a in SIM_AXES}
        p["sim_detail"] = p.get("sim_detail") or {}
        try:
            p["evidence_rank"] = max(1, min(5, int(p.get("evidence_rank", 3))))
        except Exception:
            p["evidence_rank"] = 3
        if p.get("evidence_level") not in EVIDENCE_LEVELS:
            p["evidence_level"] = EVIDENCE_LEVELS[p["evidence_rank"] - 1]
        # relevance(모델 종합 판단)이 없으면 유사도 평균으로 보정
        if p.get("relevance") is None:
            vals = list(p["sim"].values())
            p["relevance"] = round(sum(vals) / len(vals)) if vals else 60
        p["relevance"] = _clamp(p["relevance"])
        items = []
        valid_fields = {"evidence", "merits", "target", "dosage", "duration", "other", "reason2", "opinion"}
        for j, it in enumerate(p.get("items") or []):
            it["key"] = it.get("key") or f"IT{i}_{j}"
            if it.get("field") not in valid_fields:
                it["field"] = "evidence"
            items.append(it)
        p["items"] = items
        out.append(p)
    return out


def _clamp(v, lo=0, hi=100):
    try:
        return max(lo, min(hi, int(v)))
    except Exception:
        return lo


# ---------------------------------------------------------------- Flask
def create_app() -> Flask:
    app = Flask(__name__, template_folder="templates")
    CORS(app)

    @app.get("/")
    def index():
        return render_template("index.html")

    @app.get("/api/health")
    def health():
        return jsonify({"ok": True, "provider": _provider(), "model": _model(),
                        "key_present": _key_present()})

    @app.post("/api/search")
    def search():
        body = request.get_json(force=True) or {}
        pico = body.get("pico")
        text = (body.get("text") or "").strip()
        if pico:
            user = ("[PICO 입력]\n"
                    f"P(대상): {pico.get('p','')}\nI(중재): {pico.get('i','')}\n"
                    f"C(비교): {pico.get('c','')}\nO(결과): {pico.get('o','')}")
        elif text:
            user = f"[자유 서술]\n{text}"
        else:
            return jsonify({"error": "환자 상황(자유서술 또는 PICO)을 입력하세요."}), 400
        try:
            d = chat_json(SEARCH_SYS, user)
            d["papers"] = sanitize_papers(d.get("papers"))
            d["weights"] = DEFAULT_WEIGHTS
            return jsonify(d)
        except Exception as e:
            return jsonify({"error": str(e)}), 502

    @app.post("/api/drug")
    def drug():
        body = request.get_json(force=True) or {}
        name = (body.get("name") or "").strip()
        if not name:
            return jsonify({"error": "약제/질환명을 입력하세요."}), 400
        try:
            d = chat_json(DRUG_SYS, f"[{'약제' if body.get('kind','drug')=='drug' else '질환'}] {name}")
            d["papers"] = sanitize_papers(d.get("papers"))
            d["weights"] = DEFAULT_WEIGHTS
            return jsonify(d)
        except Exception as e:
            return jsonify({"error": str(e)}), 502

    return app


if __name__ == "__main__":
    port = int(os.environ.get("PORT", "8000"))
    print(f"[OncoReg Studio] http://localhost:{port}  (provider={_provider()}, model={_model()})")
    create_app().run(host="0.0.0.0", port=port, debug=False)


In [ ]:
%%writefile oncoreg_studio/templates/index.html
<!DOCTYPE html>
<html lang="ko">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>OncoReg Studio — 근거 검색 · 유사도 가중 · 문서 스튜디오</title>
<style>
:root{
  --paper:#FBFAF7;--card:#FFF;--ink:#16181A;--muted:#6B6E73;--faint:#9A9C9F;
  --rule:#E3DFD7;--rule2:#EFECE6;--seal:#0E6E5E;--seal-bg:#E8F2EF;
  --alert:#B03A2E;--alert-bg:#FBEEEC;--trial:#5A4B8C;--trial-bg:#EEEBF5;
  --amber:#9A6A00;--amber-bg:#FBF3DF;--mark:#FFF0B8;
  --mono:ui-monospace,"SF Mono",Menlo,Consolas,"D2Coding",monospace;
  --sans:-apple-system,BlinkMacSystemFont,"Pretendard","Malgun Gothic","Noto Sans KR",sans-serif;
}
*{box-sizing:border-box;margin:0;padding:0}
body{background:var(--paper);color:var(--ink);font-family:var(--sans);font-size:15px;line-height:1.6;-webkit-font-smoothing:antialiased}
.wrap{max-width:1220px;margin:0 auto;padding:0 22px 70px}
.demo{background:var(--amber-bg);border-bottom:1px solid #EAD9A8;padding:8px 22px;font-size:12px;color:#6E4E00;text-align:center}
.demo b{font-weight:600}
header{padding:22px 0 12px;border-bottom:1px solid var(--rule);margin-bottom:16px}
.brand{display:flex;align-items:baseline;gap:11px;flex-wrap:wrap}
.brand h1{font-size:19px;font-weight:600;letter-spacing:-.02em}
.brand .tag{font-size:12px;color:var(--muted);border-left:1px solid var(--rule);padding-left:11px}
.brand .eng{font-size:11px;color:#fff;background:var(--seal);padding:2px 8px;border-radius:10px;font-family:var(--mono);cursor:pointer}
.brand .eng.off{background:var(--faint)}
.tabs{display:flex;gap:8px;margin:14px 0 2px;flex-wrap:wrap}
.tabs button{border:1px solid var(--rule);background:#fff;padding:8px 15px;font-family:inherit;font-size:13.5px;color:var(--muted);cursor:pointer;border-radius:20px}
.tabs button.on{background:var(--ink);color:#fff;border-color:var(--ink)}
.tabs button:disabled{opacity:.5;cursor:not-allowed}
.view{display:none}.view.on{display:block}
.card{background:var(--card);border:1px solid var(--rule);border-radius:6px;padding:16px}
label.fl{display:block;font-size:12.5px;color:var(--muted);margin-bottom:7px}
textarea{width:100%;min-height:84px;resize:vertical;padding:11px 12px;border:1px solid var(--rule);border-radius:5px;font-family:inherit;font-size:14.5px;line-height:1.6;color:var(--ink);background:#fff}
textarea:focus,input:focus,select:focus{outline:2px solid var(--seal);outline-offset:-1px;border-color:transparent}
input.txt,select.sel{padding:9px 11px;border:1px solid var(--rule);border-radius:5px;font-family:inherit;font-size:14px;background:#fff;color:var(--ink)}
.pico{display:grid;grid-template-columns:1fr 1fr;gap:10px}
@media(max-width:640px){.pico{grid-template-columns:1fr}}
.pico .pf label{font-size:12px;color:var(--seal);font-weight:600;display:block;margin-bottom:4px}
.pico .pf input{width:100%}
.chips{display:flex;gap:7px;flex-wrap:wrap;margin-top:10px}
.chip{font-size:12px;color:var(--muted);background:#F4F1EB;border:1px solid var(--rule2);border-radius:14px;padding:5px 11px;cursor:pointer}
.chip:hover{border-color:var(--seal);color:var(--seal)}
.row{margin-top:13px;display:flex;gap:12px;align-items:center;flex-wrap:wrap}
button.go{background:var(--ink);color:#fff;border:0;border-radius:5px;padding:10px 20px;font-family:inherit;font-size:14px;font-weight:500;cursor:pointer}
button.go:hover{background:#2C2F33}button.go:disabled{background:#C9C6C0;cursor:not-allowed}
button.ghost{background:#fff;color:var(--ink);border:1px solid var(--rule);border-radius:5px;padding:8px 14px;font-family:inherit;font-size:13px;cursor:pointer}
button.ghost:hover{background:#FAF8F4}
.chk{font-size:12.5px;color:var(--muted);display:flex;align-items:center;gap:6px;cursor:pointer}
.picoBox{border:1px solid var(--rule);border-radius:6px;background:#fff;padding:12px 14px;margin:14px 0 8px}
.picoBox .pt{font-size:11px;color:var(--faint);letter-spacing:.04em;text-transform:uppercase;margin-bottom:7px}
.picoBox table{width:100%;border-collapse:collapse;font-size:13px}
.picoBox td{padding:4px 6px;vertical-align:top;border-bottom:1px solid var(--rule2)}
.picoBox td:first-child{width:130px;font-weight:600;color:var(--seal)}
.simwrap{border:1px solid var(--rule);border-radius:6px;background:#fff;padding:13px 15px;margin:12px 0}
.simwrap .sh{font-size:12px;color:var(--muted);margin-bottom:9px}.simwrap .sh b{color:var(--ink)}
.wt{display:grid;grid-template-columns:100px 1fr 30px;gap:10px;align-items:center;margin-bottom:5px}
.wt label{font-size:12px}.wt input[type=range]{width:100%;accent-color:var(--seal)}
.wt .wv{font-family:var(--mono);font-size:12px;color:var(--muted);text-align:right}
.pcard{border:1px solid var(--rule);border-radius:6px;background:#fff;margin-bottom:10px;overflow:hidden}
.pcard.top{border-color:var(--seal)}
.pcard.sel{background:#FCFEFD;box-shadow:0 0 0 2px var(--seal) inset;border-color:var(--seal)}
.selbox{float:right;font-size:11.5px;color:var(--seal);display:inline-flex;align-items:center;gap:5px;cursor:pointer}
.selbox input{accent-color:var(--seal)}
.pcard .ph{padding:12px 15px}
.pcard .pt{font-size:14.5px;font-weight:500;line-height:1.45}
.pcard .pm{font-size:11.5px;color:var(--muted);font-family:var(--mono);margin-top:4px}
.pcard .links{margin-top:6px;display:flex;gap:12px;flex-wrap:wrap}
a.ext{font-size:11.5px;color:var(--seal);text-decoration:none;font-family:var(--mono)}a.ext:hover{text-decoration:underline}
.why{font-size:12.5px;color:#33363A;margin-top:8px;background:var(--seal-bg);border-radius:4px;padding:8px 10px;line-height:1.55}
.simline{display:flex;align-items:center;gap:8px;margin-top:9px}
.simtrack{flex:1;height:6px;background:#EEEBE5;border-radius:3px;overflow:hidden}.simfill{height:100%;background:var(--seal)}
.simsc{font-family:var(--mono);font-size:13px;font-weight:600;color:var(--seal)}
.evb{display:inline-block;font-size:10.5px;font-weight:600;padding:2px 8px;border-radius:3px;margin-right:6px;vertical-align:1px}
.ev1{background:#DDEEE6;color:#0B5647}.ev2{background:#E6EFDD;color:#3B5A16}.ev3{background:#EDEBD6;color:#6E5E10}.ev4{background:#F3E7DA;color:#7A4E12}.ev5{background:#F1E4E2;color:#8A2F26}
.topbadge{display:inline-block;font-size:10.5px;color:#fff;background:var(--seal);border-radius:3px;padding:1px 7px;margin-left:6px;font-weight:600}
.det{border-top:1px solid var(--rule2);background:#FCFBF9;padding:6px 15px}
.det summary{font-size:11.5px;color:var(--muted);cursor:pointer;padding:5px 0}
.axr{display:grid;grid-template-columns:82px 60px 1fr;gap:9px;align-items:center;font-size:11.5px;padding:2px 0}
.axr .ab{height:5px;background:#EEEBE5;border-radius:3px;overflow:hidden}.axr .af{height:100%;background:var(--seal)}
.axr .ar{color:#44474B}
.items{border-top:1px solid var(--rule2);background:#FCFBF9;padding:9px 15px;display:flex;flex-wrap:wrap;gap:6px}
.itchip{font-size:11.5px;font-family:var(--mono);background:#fff;border:1px solid var(--rule);border-radius:3px;padding:3px 8px}.itchip b{color:var(--seal)}
.dcard{border:1px solid var(--rule);border-radius:6px;background:#fff;padding:15px;margin-bottom:12px}
.dcard h3{font-size:14px;font-weight:600;margin-bottom:8px}
.deslist{list-style:none}.deslist li{font-size:13px;padding:4px 0 4px 16px;position:relative;color:#33363A}
.deslist li::before{content:"–";position:absolute;left:0;color:var(--seal)}
.drugs{display:flex;flex-wrap:wrap;gap:7px;margin-top:10px}
.drug{font-size:12.5px;background:#fff;border:1px solid var(--rule);border-radius:16px;padding:6px 13px;cursor:pointer}
.drug:hover{border-color:var(--seal);color:var(--seal)}
.studio{display:grid;grid-template-columns:minmax(360px,44%) 1fr;gap:16px;align-items:start}
@media(max-width:960px){.studio{grid-template-columns:1fr}}
.pane{border:1px solid var(--rule);border-radius:6px;background:#fff;padding:13px}
.pane h3{font-size:13px;font-weight:600;margin-bottom:8px}
.ctrl{display:grid;grid-template-columns:92px 1fr;gap:7px 10px;align-items:center;font-size:12.5px;margin-bottom:5px}
.ctrl label{color:var(--muted)}
.srcview{font-size:12px;line-height:1.65;color:#44474B;background:#FCFBF9;border:1px solid var(--rule2);border-radius:4px;padding:9px;margin:8px 0;min-height:18px}
.srcview mark{background:var(--mark);padding:0 2px;border-radius:1px}
.fld{border:1px solid var(--rule2);border-radius:5px;padding:9px 10px;margin-bottom:9px}
.fld.act{border-color:var(--seal);box-shadow:0 0 0 1px var(--seal) inset}
.fld .fn{font-size:12px;font-weight:600;margin-bottom:5px}
.fld textarea{min-height:42px;font-size:13px;padding:7px 9px}
.fld .picks{display:flex;flex-wrap:wrap;gap:5px;margin-top:6px}
.pk{font-size:11px;font-family:var(--mono);background:#F4F1EB;border:1px solid var(--rule2);border-radius:3px;padding:2px 7px;cursor:pointer}
.pk:hover{border-color:var(--seal);color:var(--seal);background:#fff}.pk b{color:var(--seal)}
.viewer{position:sticky;top:12px}
.gov{color:#111;font-size:12px;line-height:1.5}
.gov .ghead{font-size:10.5px;color:#333;margin-bottom:5px}
.gov h3.gt{text-align:center;font-size:16px;font-weight:700;letter-spacing:.05em;margin:2px 0}
.gov .gsub{text-align:center;font-size:11.5px;color:#333;margin-bottom:8px}
.gov table.gf{width:100%;border-collapse:collapse;border:1.4px solid #222;table-layout:fixed}
.gov table.gf td{border:.8px solid #666;padding:4px 6px;vertical-align:top;word-break:break-word;font-size:11px}
.gov td.gl{background:#F1EEE7;font-weight:600;text-align:center}
.gov td.gl2{background:#F8F6F1;text-align:center;font-size:11px}.gov td.gl3{background:#F8F6F1;text-align:center;font-size:11px;font-weight:600}
.gov .cbi{display:inline-block;margin-right:10px;font-size:11px;line-height:1.95;font-family:var(--mono)}
.gov .oh{font-weight:600;display:block;margin-top:3px}.gov .oh:first-child{margin-top:0}
.gov .blank{color:#BBB6AC}.gov .cite{font-family:var(--mono);font-size:10px;font-weight:600;background:var(--seal-bg);color:var(--seal);padding:0 3px;border-radius:2px}
.gov .fc{cursor:pointer;display:block;min-height:15px;padding:1px 2px;border-radius:2px}
.gov .fc:hover{background:#EFF5F2;outline:1px dashed var(--seal)}
.gov .fc.on{background:var(--seal-bg);outline:1px solid var(--seal)}
.gov .fcp{color:var(--seal);font-size:10px}
.gov .gapply{margin:12px 2px 2px;text-align:center}.gov .gdate{text-align:center;letter-spacing:.1em;margin:7px 0}
.gov .gsign{text-align:right;line-height:1.9;margin:4px 26px 4px 0}.gov .gto{font-weight:600;margin:7px 2px}
.gov .fn{font-size:10px;color:#444;margin-top:8px;border-top:1px solid var(--rule);padding-top:6px}
.aibar{background:#EEF3FA;border:1px solid #CFDBEC;border-radius:5px;padding:10px 13px;font-size:12px;color:#2B4A73;margin-bottom:12px}
.okbar{background:var(--seal-bg);border:1px solid #C5DFD8;border-radius:5px;padding:10px 13px;font-size:12px;color:#0B5647;margin-bottom:12px}
.spin{display:inline-block;width:12px;height:12px;border:2px solid var(--rule);border-top-color:var(--ink);border-radius:50%;animation:sp .7s linear infinite;vertical-align:-2px;margin-right:7px}
@keyframes sp{to{transform:rotate(360deg)}}@media(prefers-reduced-motion:reduce){.spin{animation:none}}
footer{margin-top:34px;padding-top:14px;border-top:1px solid var(--rule);font-size:11px;color:var(--faint);line-height:1.7}
</style>
</head>
<body>
<div class="demo"><b>데모/프로토타입.</b> 문헌·수치·지정·가이드라인은 <b>AI가 생성한 참고 초안</b>으로 실제와 다를 수 있습니다. 제출 전 원문/규제 공고 검증 필수. 논문 링크는 제목 기반 검색 링크입니다.</div>
<div class="wrap">
<header>
  <div class="brand"><h1>OncoReg Studio</h1><span class="tag">근거 검색 · 관련도 랭킹 · 문서 스튜디오</span></div>
  <div class="tabs">
    <button id="tSearch" class="on" onclick="mode('search')">① 자유서술 검색</button>
    <button id="tPico" onclick="mode('pico')">② PICO 단답형</button>
    <button id="tDrug" onclick="mode('drug')">③ 약제 선택</button>
    <button id="tStudio" onclick="openStudio()">④ 문서 스튜디오</button>
  </div>
</header>

<!-- ① 자유서술 -->
<div class="view on" id="vSearch">
  <div class="card">
    <label class="fl">환자 상황을 자유롭게 적어주세요 (진단·유전체·증상·나이·기저질환·병리·기존 투약 등 · 식별정보 제외)</label>
    <textarea id="q" placeholder="예) 당뇨병성 만성신부전으로 투석이 필요한 62세 환자. HbA1c 8.0, 10년간 메트포르민, 최근 SGLT2 억제제 사용. 어떤 투석·약제 근거가 있는지 가장 비슷한 논문을 찾아줘."></textarea>
    <div class="chips" id="exs"></div>
    <div class="row"><button class="go" id="btnSearch" onclick="runSearch()">근거 검색</button>
      <button class="ghost" onclick="startBlank()">빈 문서로 바로 작성 →</button>
      <label class="chk"><input type="checkbox" id="useDemo"> 데모</label>
      <span id="msg" style="font-size:12.5px;color:var(--muted)"></span></div>
  </div>
  <div id="out"></div>
</div>

<!-- ② PICO 단답형 -->
<div class="view" id="vPico">
  <div class="card">
    <label class="fl">전형적인 케이스는 P·I·C·O를 단답형으로 적으면 됩니다 (특이 케이스는 ① 자유서술 사용)</label>
    <div class="pico">
      <div class="pf"><label>P — 대상 환자·질환</label><input class="txt" id="pP" placeholder="예: HER2-low·HR+ 전이성 유방암, 2차 이상"></div>
      <div class="pf"><label>I — 중재(약물/시술)</label><input class="txt" id="pI" placeholder="예: 트라스투주맙 데룩스테칸"></div>
      <div class="pf"><label>C — 비교 대상</label><input class="txt" id="pC" placeholder="예: 의사 선택 화학요법"></div>
      <div class="pf"><label>O — 평가 결과</label><input class="txt" id="pO" placeholder="예: ORR, PFS, 이상반응"></div>
    </div>
    <div class="chips" id="exsP"></div>
    <div class="row"><button class="go" id="btnPico" onclick="runPico()">근거 검색</button>
      <label class="chk"><input type="checkbox" id="useDemoP"> 데모</label>
      <span id="msgP" style="font-size:12.5px;color:var(--muted)"></span></div>
  </div>
  <div id="outP"></div>
</div>

<!-- ③ 약제 선택 -->
<div class="view" id="vDrug">
  <div class="card">
    <label class="fl">약제를 선택하면 지정(긴급승인·희귀의약품)·대상 환자군·가이드라인·근거 논문을 모아 보여줍니다 (예시 20종)</label>
    <div class="drugs" id="drugList"></div>
    <div class="row"><input class="txt" id="dName" style="flex:1;min-width:220px" placeholder="직접 입력(약제 또는 질환명)">
      <button class="go" id="btnDrug" onclick="runDrug(document.getElementById('dName').value)">조회</button>
      <label class="chk"><input type="checkbox" id="useDemoD"> 데모</label>
      <span id="msgD" style="font-size:12.5px;color:var(--muted)"></span></div>
  </div>
  <div id="outD"></div>
</div>

<!-- ④ 스튜디오 -->
<div class="view" id="vStudio">
  <div style="margin-bottom:12px;display:flex;gap:8px;flex-wrap:wrap">
    <button class="ghost" onclick="back()">← 뒤로가기</button>
    <button class="ghost" onclick="window.print()">인쇄 / PDF 저장</button>
  </div>
  <div class="studio">
    <div>
      <div class="pane"><h3>서식 유형 · 체크 항목</h3>
        <div style="font-size:11px;color:var(--muted);margin-bottom:8px">아래 항목(유형1~6·희귀질환여부·<b>질환유형</b>)은 모두 <b>실제 별지 제1호서식의 정식 항목</b>입니다. 선택하면 오른쪽 서식 체크박스에 반영됩니다.</div>
        <div class="ctrl"><label>허가초과 유형</label><select class="sel" id="cExcess" onchange="renderForm()">
          <option value="eff">효능·효과 초과</option><option value="dose">용법·용량 초과</option><option value="age">연령·대상군 초과</option></select></div>
        <div class="ctrl"><label>희귀질환</label><label class="chk"><input type="checkbox" id="cRare" onchange="renderForm()"> 예</label></div>
        <div class="ctrl"><label>병용여부</label><label class="chk"><input type="checkbox" id="cCombo" onchange="renderForm()"> 병용</label></div>
        <div class="ctrl"><label>질환유형</label><select class="sel" id="cSev" onchange="renderForm()">
          <option>생명을 위협하는 질환</option><option>사망에 이르는 질환</option><option>비가역적인 기능상실을 초래하는 질환</option><option>기타(해당없음 등)</option></select></div>
        <div class="ctrl"><label>주성분명</label><input class="txt" id="cDrug" oninput="renderForm()" placeholder="약제명"></div>
      </div>
      <div class="pane" style="margin-top:12px"><h3>근거 → 서식 칸 채우기 <span style="font-weight:400;color:var(--faint);font-size:11px">— 오른쪽 서식의 <b>네모칸을 클릭</b>(또는 아래 칸 선택)한 뒤, 근거를 누르면 값·원문 문장·위치가 들어갑니다</span></h3>
        <div class="srcview" id="srcView">근거를 누르면 원문 인용(문장·위치)이 여기에 표시됩니다.</div>
        <div id="fields"></div>
      </div>
    </div>
    <div class="viewer"><div class="pane"><div id="form" class="gov"></div></div></div>
  </div>
</div>

<footer>
  <div style="margin-bottom:9px;color:var(--muted)">이 프로토타입은 실제 서비스에서 <b>AI(대규모 언어모델)</b>가 근거 검색·관련도 판단·문서 초안을 수행할 예정입니다. 현재 엔진: <span id="engStat">확인 중…</span> · <a onclick="setBackend()" style="cursor:pointer;color:var(--seal);text-decoration:underline">백엔드 연결</a></div>
  본 화면은 창업 프로그램 제출용 프로토타입입니다. 검색·근거수준·관련도·지정·가이드라인·문서 초안은 AI로 생성되는 시연이며 실제 출판물·규제사실·서식과 다를 수 있습니다. 진단·치료를 판정하지 않으며 최종 판단·서명은 의료 전문가가 수행합니다.
</footer>
</div>

<script>
/* ---------- 상태 ---------- */
const AXES=[["genetic","유전적 동일성"],["symptom","증상"],["age","나이"],["comorbidity","기저질환"],["pathology","병리"],["priormed","기존 투약"],["evidence","근거수준"]];
const SIM_AXES=AXES.filter(a=>a[0]!=='evidence');
const DEFAULT_WEIGHTS={genetic:.22,symptom:.15,age:.10,comorbidity:.14,pathology:.17,priormed:.14,evidence:.08};
let WEIGHTS={...DEFAULT_WEIGHTS};
let LAST=null, lastInput=null, activeField='evidence', PALETTE=[], SCORE_MODE='sim', studioReady=false;
let selectedIds=new Set(), curHost='out';
let builder={fields:{evidence:'',merits:'',target:'',dosage:'',duration:'',other:'',reason2:'',opinion:''}};
const FIELD_DEFS=[['evidence','의학적 근거자료'],['merits','신청약제의 특·장점'],['target','대상 환자 기준'],
  ['dosage','용법·용량'],['duration','투여기간(투여중단 시기 포함)'],['other','기타(재투여 기준 등)'],
  ['reason2','고시 제2조 해당 사유'],['opinion','기타 의견']];

/* ---------- 백엔드 ---------- */
let API_BASE=(localStorage.getItem('studio_api')||'').replace(/\/+$/,'');
const api=p=>API_BASE?API_BASE+'/'+p:p;
function setBackend(){const c=localStorage.getItem('studio_api')||'';
  const u=prompt('LLM 백엔드 URL을 입력하세요 (Colab의 https://….trycloudflare.com 또는 http://localhost:8000).\n비우면 데모로 동작합니다.',c);
  if(u===null)return;API_BASE=u.trim().replace(/\/+$/,'');localStorage.setItem('studio_api',API_BASE);checkEngine();}
async function checkEngine(){const b=document.getElementById('engStat');if(!b)return;
  try{const r=await fetch(api('api/health'));const h=await r.json();
    b.textContent=h.key_present?'AI 연결됨':'연결됨(키 없음)';
  }catch(e){b.textContent='데모 모드(백엔드 미연결)';
    ['useDemo','useDemoP','useDemoD'].forEach(i=>{const el=document.getElementById(i);if(el)el.checked=true;});}}

/* ---------- 데모 ---------- */
function demoPapers(){return [
 {id:1,title:"HER2-low 전이성 유방암에서 항체-약물 접합체 vs 화학요법 (DESTINY-Breast04)",journal:"N Engl J Med",year:2022,n:557,doi:"10.1056/NEJMoa2203690",
  evidence_level:"무작위 대조연구(RCT)",evidence_rank:2,relevance:92,
  sim:{genetic:94,symptom:82,age:80,comorbidity:70,pathology:92,priormed:90},
  sim_detail:{genetic:"동일 HER2 IHC 1+/2+·HR 양성 집단",symptom:"전이성 증상 유사",age:"중앙연령 57~58세 유사",comorbidity:"주요 동반질환 제한 유사",pathology:"침윤성 유방암 병리 일치",priormed:"이전 항암 1~2차 경험 일치"},
  why:"암종·바이오마커(HER2-low)·이전 치료차수가 본 환자와 거의 동일한 3상 RCT라 근거·유사도 모두 높다. 다만 병용요법이 아닌 단독 비교라는 점이 다르다.",
  items:[{key:"ORR",label:"객관적 반응률",value:"52.6%",pre:"Among HR-positive patients, ",mark:"the confirmed objective response rate was 52.6%",post:" versus 16.3% with physician's-choice chemotherapy.",loc:"Results · Table 2"},
   {key:"PFS",label:"무진행생존(중앙값)",value:"10.1개월",pre:"",mark:"median progression-free survival was 10.1 months",post:" versus 5.4 months.",loc:"Results · Table 2"},
   {key:"AE",label:"간질성 폐질환(전등급)",value:"12.1%",pre:"Adjudicated drug-related ILD occurred in ",mark:"12.1% of patients (any grade)",post:", with grade 5 events in 0.8%.",loc:"Safety"}]},
 {id:2,title:"HER2 저발현 유방암 ADC 치료의 체계적 문헌고찰·메타분석",journal:"Ann Oncol",year:2023,n:null,doi:"",
  evidence_level:"체계적 문헌고찰/메타분석",evidence_rank:1,relevance:85,
  sim:{genetic:86,symptom:74,age:72,comorbidity:66,pathology:84,priormed:74},
  sim_detail:{genetic:"HER2-low 포함이나 이질적 집단",symptom:"전이성 전반",age:"연령 분포 넓음",comorbidity:"연구별 상이",pathology:"유방암으로 일치",priormed:"치료차수 혼재"},
  why:"근거수준은 최상(메타분석)이나 여러 연구를 묶어 집단이 넓어 개별 환자와의 세부 유사도는 RCT보다 낮다.",
  items:[{key:"HR",label:"통합 위험비(PFS)",value:"0.52",pre:"Pooled analysis showed a ",mark:"hazard ratio of 0.52",post:" for progression or death.",loc:"Forest plot"},
   {key:"REC",label:"결론",value:"ADC 우월",pre:"ADCs consistently ",mark:"improved outcomes over chemotherapy",post:" in HER2-low disease.",loc:"Conclusion"}]},
 {id:3,title:"진행성 유방암 진료 권고안 — 항HER2/ADC 항목",journal:"진료지침",year:2024,n:null,doi:"",
  evidence_level:"코호트 연구",evidence_rank:3,relevance:96,
  sim:{genetic:82,symptom:66,age:64,comorbidity:60,pathology:78,priormed:66},
  sim_detail:{genetic:"HER2-low 권고 대상 일치",symptom:"—",age:"—",comorbidity:"—",pathology:"유방암 일치",priormed:"2차 이상 권고"},
  why:"권고 등급 근거로 유용하나 개별 수치보다는 방향성 제시. 유사도는 중간.",
  items:[{key:"REC",label:"권고 등급",value:"Category 1",pre:"For HR+, HER2-low mBC, T-DXd is a ",mark:"Category 1 (preferred) regimen",post:" after prior therapy.",loc:"권고 요약"}]},
 {id:4,title:"실제임상 HER2-low 유방암 T-DXd 사용 경험(사례군)",journal:"Breast",year:2023,n:21,doi:"",
  evidence_level:"사례군/사례보고",evidence_rank:5,relevance:72,
  sim:{genetic:90,symptom:80,age:78,comorbidity:74,pathology:88,priormed:88},
  sim_detail:{genetic:"실제 HER2-low 환자군",symptom:"실사용 증상 유사",age:"실사용 연령 유사",comorbidity:"동반질환 포함",pathology:"유방암 일치",priormed:"다차 치료 경험 유사"},
  why:"근거수준은 낮지만 실제 임상 집단이라 세부 조건(연령·기저질환·치료력)은 오히려 매우 유사하다. 희귀·실사용 맥락에서 가치가 있다.",
  items:[{key:"RW",label:"실사용 반응률",value:"11/21명",pre:"Clinical response was observed in ",mark:"11 of 21 patients (52%)",post:".",loc:"Results"},
   {key:"AE",label:"실사용 이상반응",value:"경증 위주",pre:"Most adverse events were ",mark:"mild to moderate",post:".",loc:"Safety"}]}
];}
const DEMO={query:{P:"HER2-low·HR+ 전이성 유방암, 2차 이상",I:"트라스투주맙 데룩스테칸",C:"의사 선택 화학요법",O:"ORR·PFS·이상반응",
   condition:"HER2-low 전이성 유방암",drug:"트라스투주맙 데룩스테칸 (T-DXd)",biomarker:"HER2 IHC 1+, HR 양성",
   stage:"전이성(4기)",line:"2차 이상",excess_type:"eff",rare:false,special:false,summary_ko:"HER2 저발현·HR 양성 전이성 유방암, 표준 2차 이상 진행, T-DXd 검토"},
 weights:{...DEFAULT_WEIGHTS},papers:demoPapers()};
const DEMO_DRUG={drug:"트라스투주맙 데룩스테칸 (T-DXd)",disease:"HER2 저발현/양성 전이성 유방암",
 designation:{orphan:"일부 적응증 희귀 관련 지정 이력 — 확인 필요",emergency:"국내 신속·특례 여부 확인 필요",note:"규제 사실은 식약처/심평원 공고로 확인"},
 populations:["HER2 IHC 1+/2+(ISH-) 전이성 유방암, 표준치료 이후","HER2 양성 전이성 위암","HER2 변이 폐암(확대 검토군)"],
 guidelines:[{name:"진행성 유방암 진료 권고안",org:"국내 학회",year:2024,recommendation:"HER2-low 2차 이상 T-DXd 우선 고려"},{name:"NCCN Breast",org:"NCCN",year:2024,recommendation:"HR+/HER2-low에서 Category 1"}],
 weights:{...DEFAULT_WEIGHTS},papers:demoPapers().slice(0,3),confidence:"low"};

const DRUGS=["트라스투주맙 데룩스테칸(T-DXd)","사시투주맙 고비테칸","소토라십","아다그라십","라로트렉티닙","엔트렉티닙","셀퍼카티닙","프랄세티닙","오시머티닙","알렉티닙","올라파립","니라파립","리툭시맙","토실리주맙","벨리무맙","닌테다닙","마시텐탄","셀렉시팍","리오시구앗","에쿨리주맙"];
const EX_FREE=["당뇨병성 만성신부전 투석 62세, HbA1c 8.0, 메트포르민 10년·SGLT2i 최근. 관련 근거를 찾아줘.",
 "HER2 IHC 1+ HR 양성 전이성 유방암, 표준 2차 이상 진행, T-DXd 검토.",
 "KRAS G12C 변이 비소세포폐암, 표적치료 실패, ECOG 1."];
const EX_PICO=[["HER2-low·HR+ 전이성 유방암, 2차 이상","트라스투주맙 데룩스테칸","화학요법","ORR·PFS"],
 ["KRAS G12C 변이 비소세포폐암","소토라십","도세탁셀","PFS·반응률"]];

/* ---------- 공통 ---------- */
function mode(m){['search','pico','drug','studio'].forEach(x=>{
  document.getElementById('v'+x[0].toUpperCase()+x.slice(1)).classList.toggle('on',x===m);
  document.getElementById('t'+x[0].toUpperCase()+x.slice(1)).classList.toggle('on',x===m);});
  window.scrollTo({top:0,behavior:'smooth'});}
function back(){mode(lastInput&&lastInput.mode==='drug'?'drug':(lastInput&&lastInput.mode==='pico'?'pico':'search'));}
function evSub(p){return Math.round((6-(p.evidence_rank||3))/5*100);}
function paperScore(p){                     // Gemini가 판단한 관련도로 정렬(가중치 UI 없음)
  if(typeof p.relevance==='number')return p.relevance;
  const s=p.sim||{};const v=SIM_AXES.reduce((a,[k])=>a+(s[k]||0),0);return Math.round(v/SIM_AXES.length);}
const FIELD_LABEL={evidence:'의학적 근거자료',merits:'특·장점',target:'대상 환자 기준',dosage:'용법·용량',duration:'투여기간',other:'투여방법 기타',reason2:'고시 제2조 사유',opinion:'기타 의견'};
function fieldOf(it){
  if(it.field&&FIELD_LABEL[it.field])return it.field;
  const s=(it.key||'')+' '+(it.label||'');
  if(/특장점|특·장점|우월|기전|자리매김|역할|프로파일|권고|Category|MERIT|REC\b/i.test(s))return 'merits';
  if(/대상|선정|정의|기준|IHC|바이오마커|유전/i.test(s))return 'target';
  if(/용량|\bmg\b|용법|dose|일\s*\d/i.test(s))return 'dosage';
  if(/투여기간|지속 기간|duration|주간|개월(?!.*생존)/i.test(s))return 'duration';
  return 'evidence';}
function lineOf(it){const sent=((it.pre||'')+(it.mark||it.value||'')+(it.post||'')).trim();
  return `· ${it.label}: ${it.value} — "${sent}" (${it.journal||''}${it.loc?', '+it.loc:''}) [${it.paperId}]`;}
function sortedPapers(ps){return [...(ps||[])].map(p=>({p,sc:paperScore(p)})).sort((a,b)=>b.sc-a.sc).map(x=>x.p);}
function evClass(r){return 'ev'+Math.max(1,Math.min(5,r||3));}
function scholarUrl(p){return 'https://scholar.google.com/scholar?q='+encodeURIComponent((p.title||'')+' '+(p.journal||''));}
function pubmedUrl(p){return 'https://pubmed.ncbi.nlm.nih.gov/?term='+encodeURIComponent(p.title||'');}
function esc(s){return (s||'').replace(/&/g,'&amp;').replace(/</g,'&lt;');}

function renderExamples(){
  document.getElementById('exs').innerHTML=EX_FREE.map((e,i)=>`<span class="chip" onclick="document.getElementById('q').value=EX_FREE[${i}]">${e.slice(0,30)}…</span>`).join('');
  document.getElementById('exsP').innerHTML=EX_PICO.map((e,i)=>`<span class="chip" onclick="fillPico(${i})">${e[0].slice(0,20)}… / ${e[1]}</span>`).join('');
  document.getElementById('drugList').innerHTML=DRUGS.map(d=>`<span class="drug" onclick="runDrug('${d.replace(/'/g,"\\'")}')">${d}</span>`).join('');}
function fillPico(i){const e=EX_PICO[i];['P','I','C','O'].forEach((k,j)=>document.getElementById('p'+k).value=e[j]);}

/* ---------- 검색(자유/PICO) ---------- */
async function runSearch(){const text=document.getElementById('q').value.trim();
  if(!text){document.getElementById('msg').textContent="상황을 입력하세요.";return;}
  lastInput={mode:'search'};
  await doSearch({text},document.getElementById('btnSearch'),document.getElementById('msg'),document.getElementById('useDemo').checked,'out');}
async function runPico(){const p={p:document.getElementById('pP').value.trim(),i:document.getElementById('pI').value.trim(),c:document.getElementById('pC').value.trim(),o:document.getElementById('pO').value.trim()};
  if(!p.p&&!p.i){document.getElementById('msgP').textContent="최소 P·I를 입력하세요.";return;}
  lastInput={mode:'pico'};
  await doSearch({pico:p},document.getElementById('btnPico'),document.getElementById('msgP'),document.getElementById('useDemoP').checked,'outP');}
async function doSearch(payload,btn,msg,useDemo,host){
  SCORE_MODE='sim';
  if(useDemo){LAST=JSON.parse(JSON.stringify(DEMO));WEIGHTS={...DEFAULT_WEIGHTS,...(LAST.weights||{})};renderResults(host,false);return;}
  btn.disabled=true;msg.innerHTML='<span class="spin"></span>LLM으로 해석·근거 정리 중…';
  try{const r=await fetch(api('api/search'),{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify(payload)});
    if(!r.ok){const j=await r.json().catch(()=>({}));throw new Error(j.error||('HTTP '+r.status));}
    LAST=await r.json();WEIGHTS={...DEFAULT_WEIGHTS,...(LAST.weights||{})};btn.disabled=false;msg.textContent="";renderResults(host,true);
  }catch(e){btn.disabled=false;msg.innerHTML='⚠ 실패 — 데모로 표시 ('+e.message+')';LAST=JSON.parse(JSON.stringify(DEMO));WEIGHTS={...DEFAULT_WEIGHTS};renderResults(host,false);}}

function renderResults(host,isAI){
  const q=LAST.query||{};
  const pico=`<div class="picoBox"><div class="pt">PICO 단답 요약</div><table>
    <tr><td>P 대상</td><td>${esc(q.P||q.condition||'')}</td></tr>
    <tr><td>I 중재</td><td>${esc(q.I||q.drug||'')}</td></tr>
    <tr><td>C 비교</td><td>${esc(q.C||'')}</td></tr>
    <tr><td>O 결과</td><td>${esc(q.O||'')}</td></tr></table>
    <div style="font-size:11.5px;color:var(--muted);margin-top:6px">${q.special?'특이 케이스':'전형 케이스'} · ${esc(q.summary_ko||'')}</div></div>`;
  selectedIds=new Set((LAST.papers||[]).map(p=>p.id));   // 기본: 모두 선택
  document.getElementById(host).innerHTML=
    (isAI?'<div class="aibar">AI가 <b>스스로 관련도를 판단해</b> 정렬했습니다(가중치 조정 없음). 근거수준·수치·링크는 실제와 다를 수 있으니 원문 검증이 필요합니다.</div>':'<div class="okbar">데모 데이터로 표시 중입니다.</div>')
    +pico
    +`<div style="font-size:12.5px;color:var(--muted);margin:8px 2px">논문 카드를 <b>클릭해 선택/해제</b>하세요. 선택한 논문의 근거만 문서 작업에 쓰입니다.</div>`
    +`<div id="papers_${host}"></div>`
    +`<div class="row"><button class="go" id="toStudioBtn" onclick="toStudio()">선택한 논문으로 서류 작성 →</button><button class="ghost" onclick="reSearch()">다시 검색(다른 논문)</button></div>`;
  renderPapers(host);}
function renderPapers(host){const ps=sortedPapers(LAST.papers);curHost=host;
  document.getElementById('papers_'+host).innerHTML=ps.map((p,i)=>paperCard(p,i)).join('');
  updateSelBtn();}
function updateSelBtn(){const b=document.getElementById('toStudioBtn');if(b)b.textContent='선택한 논문 '+selectedIds.size+'편으로 서류 작성 →';}
function toggleSel(id){if(selectedIds.has(id))selectedIds.delete(id);else selectedIds.add(id);renderPapers(curHost);}
function paperCard(p,i){const sc=paperScore(p);const sel=selectedIds.has(p.id);
  const axr=SIM_AXES.map(([a,l])=>`<div class="axr"><span>${l}</span><span class="ab"><span class="af" style="width:${p.sim?p.sim[a]:0}%"></span></span><span class="ar">${p.sim?p.sim[a]:0} · ${esc((p.sim_detail||{})[a]||'-')}</span></div>`).join('');
  return `<div class="pcard${sel?' sel':''}${i===0?' top':''}"><div class="ph">
    <label class="selbox"><input type="checkbox" ${sel?'checked':''} onclick="event.stopPropagation();toggleSel(${p.id})"> 문서에 사용</label>
    <span class="evb ${evClass(p.evidence_rank)}">${p.evidence_level}</span><span class="pt">${esc(p.title)}${i===0?'<span class="topbadge">관련도 최상</span>':''}</span>
    <div class="pm">${esc(p.journal)} · ${p.year||''}${p.n?' · n='+p.n:''}</div>
    <div class="links"><a class="ext" href="${pubmedUrl(p)}" target="_blank" rel="noopener">PubMed 검색 ↗</a><a class="ext" href="${scholarUrl(p)}" target="_blank" rel="noopener">Google Scholar ↗</a>${p.doi?`<a class="ext" href="https://doi.org/${p.doi}" target="_blank" rel="noopener">DOI ↗</a>`:''}</div>
    ${p.why?`<div class="why"><b>왜 ${i===0?'가장 ':''}적합한가:</b> ${esc(p.why)}</div>`:''}
    <div class="simline"><div class="simtrack"><div class="simfill" style="width:${sc}%"></div></div><span class="simsc">관련도 ${sc}</span></div></div>
    <details class="det"><summary>유사도 상세 (6축) 보기</summary>${axr}</details>
    <div class="items">${(p.items||[]).map(it=>`<span class="itchip"><b>${esc(it.value)}</b> ${esc(it.label)}</span>`).join('')||'<span style="color:var(--faint);font-size:11.5px">항목 없음</span>'}</div></div>`;}
function reSearch(){if(!lastInput)return;
  if(lastInput.mode==='pico')runPico();else runSearch();}

/* ---------- ③ 약제 ---------- */
async function runDrug(name){name=(name||'').trim();if(!name){document.getElementById('msgD').textContent="약제를 선택/입력하세요.";return;}
  lastInput={mode:'drug'};mode('drug');
  const b=document.getElementById('btnDrug'),m=document.getElementById('msgD');
  if(document.getElementById('useDemoD').checked){renderDrug(JSON.parse(JSON.stringify(DEMO_DRUG)),false);return;}
  b.disabled=true;m.innerHTML='<span class="spin"></span>지정·가이드라인·근거 정리 중…';
  try{const r=await fetch(api('api/drug'),{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify({name,kind:'drug'})});
    if(!r.ok){const j=await r.json().catch(()=>({}));throw new Error(j.error||('HTTP '+r.status));}
    b.disabled=false;m.textContent="";renderDrug(await r.json(),true);
  }catch(e){b.disabled=false;m.innerHTML='⚠ 실패 — 데모로 표시 ('+e.message+')';renderDrug(JSON.parse(JSON.stringify(DEMO_DRUG)),false);}}
function renderDrug(d,isAI){SCORE_MODE='sim';
  LAST={query:{P:d.disease,I:d.drug,condition:d.disease,drug:d.drug,populations:d.populations||[],rare:/희귀/.test(JSON.stringify(d.designation||{})),excess_type:'eff',special:false},papers:d.papers||[]};
  selectedIds=new Set((d.papers||[]).map(p=>p.id));
  const des=d.designation||{};
  document.getElementById('outD').innerHTML=
    (isAI?'<div class="aibar">AI 정리 참고 정보. <b>규제 지정(긴급승인·희귀의약품)은 반드시 식약처/심평원 공고로 확인</b>하세요. (confidence: '+(d.confidence||'-')+')</div>':'<div class="okbar">데모 데이터로 표시 중입니다.</div>')
    +`<div class="dcard"><h3>${esc(d.drug||'')} <span style="font-weight:400;color:var(--muted);font-size:12px">· ${esc(d.disease||'')}</span></h3>
      <ul class="deslist"><li><b>희귀의약품 지정:</b> ${esc(des.orphan||'확인 필요')}</li><li><b>긴급·신속승인/특례:</b> ${esc(des.emergency||'확인 필요')}</li>${des.note?`<li style="color:var(--muted)">${esc(des.note)}</li>`:''}</ul></div>
    <div class="dcard"><h3>고려 대상 환자군 <span style="font-weight:400;color:var(--faint);font-size:11px">— 클릭하면 서류의 '대상 환자 기준'에 넣습니다</span></h3>
      <ul class="deslist">${(d.populations||[]).map((x,idx)=>`<li><span class="chip" style="margin:0" onclick="pickPop(${idx})">${esc(x)}</span></li>`).join('')||'<li>정보 없음</li>'}</ul></div>
    <div class="dcard"><h3>가이드라인 · 권고</h3><ul class="deslist">${(d.guidelines||[]).map(g=>`<li><b>${esc(g.name)}</b> (${esc(g.org)}${g.year?', '+g.year:''}) — ${esc(g.recommendation)}</li>`).join('')||'<li>정보 없음</li>'}</ul></div>
    <div style="font-size:12px;color:var(--muted);margin:6px 2px">환자 정보가 없어 <b>약제 관련도순(AI 판단)</b>으로 정렬합니다. 특정 환자에 맞추려면 ① 자유서술/② PICO 를 쓰세요. 논문 카드를 클릭해 선택하세요.</div>
    <div id="papers_outD"></div>
    <div class="row"><button class="go" id="toStudioBtn" onclick="toStudio()">선택한 논문으로 서류 작성 →</button></div>`;
  renderPapers('outD');}
let _drugPops=[];function pickPop(i){const q=(LAST&&LAST.query)||{};_drugPops=q.populations||[];const v=_drugPops[i]||'';
  builder.fields.target=(builder.fields.target?builder.fields.target+' · ':'')+v; if(document.getElementById('form').innerHTML){renderFields();renderForm();}
  const b=event&&event.target;if(b){const o=b.textContent;b.textContent='서류에 추가됨 ✓';setTimeout(()=>b.textContent=o,1200);}}

/* ---------- ④ 스튜디오 ---------- */
function toStudio(){
  const q=(LAST&&LAST.query)||{};
  document.getElementById('cExcess').value=q.excess_type||'eff';
  document.getElementById('cRare').checked=!!q.rare;
  document.getElementById('cCombo').checked=/병용/.test((q.line||'')+' '+(q.biomarker||''));
  document.getElementById('cDrug').value=q.drug||q.I||'';
  builder.fields={evidence:'',merits:'',target:'',dosage:'',duration:'',other:'',reason2:'',opinion:''};
  builder.fields.target=[q.condition||q.P,q.stage,q.line].filter(Boolean).join(' · ')+(q.biomarker?(' ('+q.biomarker+')'):'');
  // 선택한 논문의 근거만 팔레트로. 문서는 '비운 채' 시작 → 칸을 클릭해 채운다.
  const src=((LAST&&LAST.papers)||[]).filter(p=>selectedIds.size===0||selectedIds.has(p.id));
  PALETTE=[];sortedPapers(src).forEach(p=>(p.items||[]).forEach(it=>PALETTE.push({...it,paperId:p.id,journal:p.journal})));
  activeField='evidence';studioReady=true;document.getElementById('tStudio').disabled=false;
  renderFields();renderForm();mode('studio');}
function openStudio(){ if(!studioReady){toStudio();} else {renderFields();renderForm();mode('studio');} }
function startBlank(){LAST=null;PALETTE=[];SCORE_MODE='sim';toStudio();}
function autoDraft(){
  const by={};PALETTE.forEach(it=>{const k=fieldOf(it);(by[k]=by[k]||[]).push(it);});
  FIELD_DEFS.forEach(([k])=>{ if(k==='target')return;   // 대상은 질환 조건에서 이미 채움
    const items=by[k]||[]; if(items.length)builder.fields[k]=items.map(lineOf).join('\n'); });}
function clearFields(){FIELD_DEFS.forEach(([k])=>builder.fields[k]='');renderFields();renderForm();}
function renderFields(){
  const host=document.getElementById('fields');if(!host)return;
  const tools=`<div class="row" style="margin:0 0 8px"><button class="ghost" onclick="autoDraft();renderFields();renderForm()">AI 초안 채우기(칸별 자동배치)</button><button class="ghost" onclick="clearFields()">모두 비우기</button></div>`;
  const flds=FIELD_DEFS.map(([k,l])=>{
    const its=PALETTE.map((it,idx)=>({it,idx})).filter(x=>fieldOf(x.it)===k);
    const chips=its.map(({it,idx})=>`<span class="pk" title="${esc(it.label)} · ${esc(it.journal||'')} ${esc(it.loc||'')}" onclick="insertItem('${k}',${idx})"><b>${esc(it.value)}</b> ${esc(it.label)}</span>`).join('');
    return `<div class="fld${activeField===k?' act':''}" id="fld_${k}"><div class="fn">${l}</div>
      <textarea onfocus="activeField='${k}';mark()" oninput="builder.fields['${k}']=this.value;renderForm()" placeholder="직접 입력하거나 아래 근거를 눌러 넣으세요">${esc(builder.fields[k])}</textarea>
      <div class="picks">${chips||'<span style="color:var(--faint);font-size:11px">이 칸에 배정된 근거 없음 — 직접 입력하세요</span>'}</div></div>`;}).join('');
  const all=PALETTE.length?`<details class="det" style="margin-top:6px"><summary>전체 근거에서 직접 고르기(활성 칸으로)</summary>
    <div style="font-size:11px;color:var(--muted);margin:6px 0">넣을 칸 입력창을 먼저 클릭(활성화)한 뒤 아래를 누르세요.</div>
    ${PALETTE.map((it,idx)=>`<span class="pk" onclick="insertItem(activeField,${idx})"><b>${esc(it.value)}</b> ${esc(it.label)} <span style="color:var(--faint)">→${FIELD_LABEL[fieldOf(it)]}</span></span>`).join('')}</details>`:'';
  host.innerHTML=tools+flds+all;}
function mark(){FIELD_DEFS.forEach(([k])=>{const el=document.getElementById('fld_'+k);if(el)el.classList.toggle('act',k===activeField);});}
function insertItem(k,idx){const it=PALETTE[idx];if(!it)return;activeField=k;
  const line=lineOf(it);
  if(!(builder.fields[k]||'').includes(line))
    builder.fields[k]=(builder.fields[k]?builder.fields[k].replace(/\s*$/,'')+'\n':'')+line;
  document.getElementById('srcView').innerHTML=`<b>[${it.paperId}] ${esc(it.journal||'')} · ${esc(it.loc||'')}</b><br>${esc(it.pre||'')}<mark>${esc(it.mark||it.value)}</mark>${esc(it.post||'')}`;
  renderFields();renderForm();const ta=document.querySelector('#fld_'+k+' textarea');if(ta){ta.focus();ta.selectionStart=ta.value.length;}}

function cbx(on,t){return '<span class="cbi">'+(on?'[✔]':'[  ]')+' '+t+'</span>';}
function nl(s){return esc(s).replace(/\n/g,'<br>');}
function focusField(k){activeField=k;renderFields();renderForm();
  const el=document.getElementById('fld_'+k);if(el){el.scrollIntoView({behavior:'smooth',block:'center'});const ta=el.querySelector('textarea');if(ta)ta.focus();}}
function renderForm(){
  const c={excess:document.getElementById('cExcess').value,rare:document.getElementById('cRare').checked,combo:document.getElementById('cCombo').checked,sev:document.getElementById('cSev').value};
  const f=builder.fields, drug=document.getElementById('cDrug').value||'', B='<span class="blank">　　</span>';
  const t1eff=c.excess==='eff',t1dose=c.excess==='dose',t1age=c.excess==='age',t2sel=c.rare?0:2,sev=c.sev;
  // 서식 칸을 클릭하면 왼쪽 편집기가 활성화되고 그 칸에 근거를 넣을 수 있다.
  const fc=(k,v)=>`<span class="fc${activeField===k?' on':''}" onclick="focusField('${k}')">${v?nl(v):'<span class="fcp">＋ 근거 선택/입력</span>'}</span>`;
  document.getElementById('form').innerHTML=`
   <div class="ghead">■ 허가 또는 신고범위 초과 약제 비급여 사용승인에 관한 기준 및 절차 [별지 제1호서식]</div>
   <h3 class="gt">허가초과 약제 비급여 사용승인 신청서</h3><div class="gsub">(IRB 지정 요양기관)</div>
   <table class="gf"><colgroup><col style="width:14.4%"><col style="width:8.4%"><col style="width:13.9%"><col style="width:13.7%"><col style="width:19.9%"><col style="width:29.7%"></colgroup>
    <tr><td class="gl">요양기관명칭</td><td colspan="3">${B}</td><td class="gl">요양기호</td><td>${B}</td></tr>
    <tr><td class="gl">주성분명<br>(주성분코드)</td><td colspan="3">${nl(drug)||B} (　)</td><td class="gl">제품명<br>(제품코드)</td><td>(　)</td></tr>
    <tr><td class="gl" rowspan="10">허가초과<br>(중복기재<br>가능)</td>
        <td class="gl2" rowspan="2">유형1</td><td colspan="4">${cbx(t1eff,'효능‧효과 초과')}${cbx(t1dose,'용법‧용량 초과')}${cbx(t1age,'연령‧대상군 초과')}</td></tr>
    <tr><td colspan="4">${cbx(false,'기타 <　>')}</td></tr>
    <tr><td class="gl2" rowspan="3">유형2</td><td colspan="4">${cbx(t2sel===0,'대체약제가 없는 경우')}${cbx(false,'대체약제가 있으나 투여금기인 경우')}</td></tr>
    <tr><td colspan="4">${cbx(t2sel===2,'대체약제보다 비용 효과적이거나 부작용이 적고 치료효과가 높을 것으로 기대되는 경우')}</td></tr>
    <tr><td colspan="4">${cbx(false,'기타 <　>')}</td></tr>
    <tr><td class="gl2">유형3</td><td colspan="4">${cbx(true,'성인')}${cbx(false,'소아')}${cbx(false,'임산부')}</td></tr>
    <tr><td class="gl2">유형4</td><td class="gl3">약제 병용여부</td><td colspan="3">${cbx(!c.combo,'단독')}${cbx(c.combo,'병용<약제: '+(c.combo?esc(drug):'　')+'>')}</td></tr>
    <tr><td class="gl2">유형5</td><td class="gl3">희귀질환여부</td><td colspan="3">${cbx(c.rare,'예(희귀질환 근거 기재)')}${cbx(!c.rare,'아니오')}</td></tr>
    <tr><td class="gl2" rowspan="2">유형6</td><td class="gl3" rowspan="2">질환유형</td><td colspan="3">${cbx(sev==='사망에 이르는 질환','사망에 이르는 질환')}${cbx(sev==='생명을 위협하는 질환','생명을 위협하는 질환')}</td></tr>
    <tr><td colspan="3">${cbx(sev==='비가역적인 기능상실을 초래하는 질환','비가역적인 기능상실을 초래하는 질환')}${cbx(sev==='기타(해당없음 등)','기타(해당없음 등)')}</td></tr>
    <tr><td class="gl" colspan="2">IRB 심사일자</td><td colspan="4">${B}</td></tr>
    <tr><td class="gl" colspan="2">IRB 심사내용</td><td colspan="4">${B}</td></tr>
    <tr><td class="gl" rowspan="5">제출자료<br>(요약)</td>
        <td class="gl2" rowspan="2">의학적<br>근거</td><td colspan="4"><span class="oh">○ 의학적 근거자료</span>${fc('evidence',f.evidence)}</td></tr>
    <tr><td colspan="4"><span class="oh">○ 신청약제의 특‧장점</span>${fc('merits',f.merits)}</td></tr>
    <tr><td class="gl2">투여<br>대상</td><td colspan="4"><span class="oh">○ 대상 환자 기준</span>${fc('target',f.target)}</td></tr>
    <tr><td class="gl2">투여<br>방법</td><td colspan="4"><span class="oh">○ 용법‧용량</span>${fc('dosage',f.dosage)}<span class="oh">○ 투여기간(투여중단 시기 포함)</span>${fc('duration',f.duration)}<span class="oh">○ 기타(재투여 기준 등)</span>${fc('other',f.other)}</td></tr>
    <tr><td class="gl2">기타</td><td colspan="4"><span class="oh">○ 신청약제 소요비용</span><span class="blank">요양기관 작성</span><span class="oh">○ 고시 제2조 각 호 해당 사유</span>${fc('reason2',f.reason2)}</td></tr>
    <tr><td class="gl" colspan="2">기타 의견</td><td colspan="4">${fc('opinion',f.opinion)}</td></tr>
   </table>
   <div class="gapply">허가 또는 신고범위 초과 약제 비급여 사용승인에 관한 기준 및 절차에 따라 위와 같이 비급여 사용승인을 신청합니다.</div>
   <div class="gdate">　　　年　　月　　日</div>
   <div class="gsign"><div>요양기관의 장　　　(서명 또는 인)</div><div>작성자</div><div>연락처　　　E-mail</div></div>
   <div class="gto">건강보험심사평가원장 귀하</div>
   <div class="fn"><b>작성방법</b> 1. IRB: 의약품임상시험실시기관 2. 각 자료는 별첨. 3. 약제정보·가이드라인·학술지 수재내역 등 기재.</div>`;}

renderExamples();checkEngine();
</script>
</body>
</html>


## 3) API 키 입력 (Gemini 또는 OpenAI · 직접 입력 칸)
- Gemini 키: <https://aistudio.google.com/app/apikey>  ·  OpenAI 키: <https://platform.openai.com/api-keys>
- 공급자를 고르고 키를 붙여넣으세요. (보안 비밀 `GEMINI_API_KEY`/`OPENAI_API_KEY` 가 있으면 자동으로 채워집니다)
- **모델(Model 칸)**: 빠름 `gemini-2.5-flash-lite` / `gemini-2.0-flash` · 균형 `gemini-2.5-flash`(기본) · 고품질(느림) `gemini-2.5-pro`.
- **속도 팁**: 2.5 flash 계열은 기본 '사고(thinking)'로 느릴 수 있어, 서버는 `GEMINI_THINKING=0`(사고 끔, 기본)으로 빠르게 돕니다. 의학 추론 품질을 더 원하면 아래 실행 셀 전에 `os.environ['GEMINI_THINKING']='512'` 처럼 올리거나 모델을 `gemini-2.5-pro`로 바꾸세요(느려짐).

In [ ]:
import os
_pre_g=_pre_o=''
try:
    from google.colab import userdata
    _pre_g=userdata.get('GEMINI_API_KEY') or ''
    _pre_o=userdata.get('OPENAI_API_KEY') or ''
except Exception:
    pass

def _verify():
    prov=os.environ.get('LLM_PROVIDER','gemini')
    try:
        if prov=='gemini':
            from google import genai
            genai.Client(api_key=os.environ.get('GEMINI_API_KEY','')).models.generate_content(
                model=os.environ.get('GEMINI_MODEL','gemini-2.5-flash'), contents='ping')
            print('✅ Gemini 연결 OK · 모델:', os.environ.get('GEMINI_MODEL'))
        else:
            from openai import OpenAI
            OpenAI(api_key=os.environ.get('OPENAI_API_KEY','')).chat.completions.create(
                model=os.environ.get('OPENAI_MODEL','gpt-4o-mini'),
                messages=[{'role':'user','content':'ping'}], max_tokens=1)
            print('✅ OpenAI 연결 OK · 모델:', os.environ.get('OPENAI_MODEL'))
    except Exception as e:
        print('⚠️ 연결 확인만 실패(키가 맞아도 날 수 있음):', e)
        print('   → 키를 칸에 제대로 넣었다면 4번 서버 실행 셀로 진행해도 됩니다.')

try:
    import ipywidgets as w
    from IPython.display import display
    _prov=w.Dropdown(options=[('Gemini','gemini'),('OpenAI','openai')], value='gemini',
                     description='공급자', style={'description_width':'70px'})
    _key=w.Text(value=_pre_g, description='API Key', placeholder='키 붙여넣기',
                layout=w.Layout(width='620px'), style={'description_width':'70px'})
    _model=w.Text(value='gemini-2.5-flash', description='Model',
                  layout=w.Layout(width='360px'), style={'description_width':'70px'})
    _btn=w.Button(description='저장하고 확인', button_style='success'); _out=w.Output()
    def _on_prov(ch):
        if ch['new']=='gemini': _key.value=_pre_g; _model.value='gemini-2.5-flash'
        else: _key.value=_pre_o; _model.value='gpt-4o-mini'
    _prov.observe(_on_prov, names='value')
    def _save(_):
        with _out:
            _out.clear_output()
            prov=_prov.value; os.environ['LLM_PROVIDER']=prov
            k=_key.value.strip()
            if not k: print('⚠️ 키 칸이 비어 있어요.'); return
            if prov=='gemini':
                os.environ['GEMINI_API_KEY']=k; os.environ['GEMINI_MODEL']=_model.value.strip() or 'gemini-2.5-flash'
            else:
                os.environ['OPENAI_API_KEY']=k; os.environ['OPENAI_MODEL']=_model.value.strip() or 'gpt-4o-mini'
            print('저장됨('+prov+'). 확인 중…'); _verify()
    _btn.on_click(_save); display(w.VBox([_prov, _key, w.HBox([_model,_btn]), _out]))
    print('↑ 공급자 선택 → 키 붙여넣고 [저장하고 확인].')
except Exception:
    os.environ['LLM_PROVIDER']='gemini'
    os.environ['GEMINI_API_KEY']=input('Gemini API Key 붙여넣고 Enter: ').strip()
    os.environ.setdefault('GEMINI_MODEL','gemini-2.5-flash'); _verify()


## 4) 서버 실행 → 공개 URL
출력되는 `https://….trycloudflare.com` 주소를 새 탭에서 열면 스튜디오가 뜹니다. 이 셀은 계속 실행 상태로 둡니다(중지: ⏹️).

In [ ]:
import sys
sys.path.insert(0,'oncoreg_studio'); sys.modules.pop('studio_app',None)
from studio_app import create_app
from flask_cloudflared import run_with_cloudflared
app=create_app(); print('>>> 실행 중인 앱: OncoReg Studio (studio_app, port 8020)')
run_with_cloudflared(app); app.run(port=8020)


### (대안) cloudflared 가 안 될 때 — Colab 내장 프록시

In [ ]:
import sys, threading
sys.path.insert(0,'oncoreg_studio'); sys.modules.pop('studio_app',None)
from studio_app import create_app
app=create_app()
threading.Thread(target=lambda: app.run(port=8020, use_reloader=False), daemon=True).start()
from google.colab.output import serve_kernel_port_as_window
serve_kernel_port_as_window(8020)
